In [ ]:
pip install ollama

In [ ]:
pip install langchain langchain-openai pydantic python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 49.3 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.28
    Uninstalling langchain-core-1.2.28:
      Successfully uninstalled langchain-core-1.2.28


In [ ]:
!sudo apt-get install zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 42 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 2s (354 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 1.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package zstd.
(Reading database ... 122354 files and directories currently i

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
import ollama
import json
import time
import math
import csv
import os
import random
from collections import Counter
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

In [ ]:
%%bash --bg
ollama serve

In [ ]:
!mkdir -p /content/drive/MyDrive/ollama_models
!rm -rf ~/.ollama/models
!mkdir -p ~/.ollama
!ln -s /content/drive/MyDrive/ollama_models ~/.ollama/models

In [ ]:
!rm -rf ~/.ollama/models
!mkdir -p /content/ollama_models
!ln -s /content/ollama_models ~/.ollama/models


In [ ]:
import subprocess
import time

In [ ]:
process = subprocess.Popen(["ollama", "serve"])

In [ ]:
!ollama pull mistral-nemo:12b
!ollama pull qwen2:7b
!ollama pull llama3.1:8b
!ollama pull gemma2:9b
!ollama pull deepseek-r1:8b
!ollama pull phi3:3.8b


In [ ]:
!ollama list

NAME                ID              SIZE      MODIFIED               
phi3:3.8b           4f2222927938    2.2 GB    Less than a second ago    
deepseek-r1:8b      6995872bfe4c    5.2 GB    19 seconds ago            
gemma2:9b           ff02c3702f32    5.4 GB    59 seconds ago            
llama3.1:8b         46e0c10c039e    4.9 GB    About a minute ago        
qwen2:7b            dd314f039b9d    4.4 GB    2 minutes ago             
mistral-nemo:12b    e7e06d107c6c    7.1 GB    3 minutes ago             


In [ ]:


# =============================
# CONFIG
# =============================

TEAM = [
    "mistral-nemo:12b",
    "qwen2:7b",
    "llama3.1:8b",
    'phi3:3.8b',
    'deepseek-r1:8b'
]



CAPTAIN_MODEL = "gemma2:9b"

MAX_WORKERS = len(TEAM)

LOG_FILE = "debate_captian_experiment_log.jsonl"

SYSTEM_PROMPT = "You are a precise trivia expert. Answer concisely."

# =============================
# UTIL
# =============================

def normalize(text):
    return text.lower().strip()

def load_existing():
    results = {}
    if not os.path.exists(LOG_FILE):
        return results

    with open(LOG_FILE, "r") as f:
        for line in f:
            entry = json.loads(line)
            results[entry["question_id"]] = entry

    return results

def save_entry(entry):
    with open(LOG_FILE, "a") as f:
        f.write(json.dumps(entry) + "\n")

def get_last_completed_index():
    existing = load_existing()
    if not existing:
        return 0
    return max(existing.keys()) + 1

# =============================
# MODEL CALL
# =============================

def ask_model(model, prompt, temperature=0.2):
    start = time.time()

    response = ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        options={"temperature": temperature}
    )

    latency = time.time() - start
    answer = response["message"]["content"].strip()

    return {
        "model": model,
        "answer_raw": answer,
        "answer_norm": normalize(answer),
        "latency": latency
    }

# =============================
# PARALLEL TEAM
# =============================

def ask_team_parallel(prompt):
    results = []

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {
            executor.submit(ask_model, model, prompt): model
            for model in TEAM
        }

        for future in as_completed(futures):
            results.append(future.result())

    return results

# =============================
# CAPTAIN DECISION
# =============================

def captain_decide(question, answers_round2):

    # Shuffle to reduce ordering bias
    shuffled = answers_round2.copy()
    random.shuffle(shuffled)

    answers_text = "\n".join(
        f"{a['model']}: {a['answer_raw']}"
        for a in shuffled
    )

    captain_prompt = f"""
You are the captain.

Question:
{question}

Team answers:
{answers_text}

Select the best answer from the team OR provide a better one if needed.

Respond in JSON format:
{{
    "final_answer": "...",
    "choice_type": "selected" or "generated",
    "selected_model": "model_name" or null
}}
"""

    response = ollama.chat(
        model=CAPTAIN_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": captain_prompt}
        ],
        options={"temperature": 0}
    )

    raw = response["message"]["content"]

    try:
        parsed = json.loads(raw)

        final_answer = normalize(parsed["final_answer"])
        choice_type = parsed.get("choice_type", "unknown")
        selected_model = parsed.get("selected_model", None)

    except:
        # fallback if parsing fails
        final_answer = normalize(raw)
        choice_type = "fallback"
        selected_model = None

    return {
        "captain_answer": final_answer,
        "captain_raw": raw,
        "choice_type": choice_type,
        "selected_model": selected_model
    }

# =============================
# RUN QUESTION
# =============================

def run_question(question, true_answer=None):

    # -------------------------
    # ROUND 1
    # -------------------------
    answers_round1 = ask_team_parallel(question)

    answers_text = "\n".join(
        f"{a['model']}: {a['answer_raw']}"
        for a in answers_round1
    )

    # -------------------------
    # ROUND 2 (DEBATE)
    # -------------------------
    debate_prompt = f"""
Question:
{question}

Other models answered:
{answers_text}

Consider the other answers carefully.
If another answer seems more correct, you may change your answer.

Give your FINAL answer only.
"""

    answers_round2 = ask_team_parallel(debate_prompt)

    # -------------------------
    # MAJORITY VOTE (for analysis)
    # -------------------------
    votes = Counter(a["answer_norm"] for a in answers_round2)

    majority_answer = None
    max_votes = max(votes.values())
    top_answers = [a for a, c in votes.items() if c == max_votes]

    if len(top_answers) == 1:
        majority_answer = top_answers[0]

    agreement_rate = max_votes / len(TEAM)

    entropy = -sum(
        (c/len(TEAM)) * math.log2(c/len(TEAM))
        for c in votes.values()
    )

    # -------------------------
    # CAPTAIN DECISION
    # -------------------------
    captain_result = captain_decide(question, answers_round2)

    final_answer = captain_result["captain_answer"]

    # -------------------------
    # CHANGE RATE
    # -------------------------
    change_rate = sum(
        r1["answer_norm"] != r2["answer_norm"]
        for r1, r2 in zip(
            sorted(answers_round1, key=lambda x: x["model"]),
            sorted(answers_round2, key=lambda x: x["model"])
        )
    ) / len(TEAM)

    # -------------------------
    # ENTRY
    # -------------------------
    entry = {
        "question_id": None,
        "timestamp": datetime.utcnow().isoformat(),
        "question": question,
        "true_answer": true_answer,

        "round1_answers": answers_round1,
        "round2_answers": answers_round2,

        "vote_distribution": dict(votes),
        "majority_answer": majority_answer,
        "agreement_rate": agreement_rate,
        "entropy": entropy,
        "change_rate": change_rate,

        "captain_answer": captain_result["captain_answer"],
        "captain_choice_type": captain_result["choice_type"],
        "captain_selected_model": captain_result["selected_model"],

        "final_answer": final_answer
    }

    if true_answer:
        entry["correct_majority"] = normalize(true_answer) == (majority_answer or "")
        entry["correct_captain"] = normalize(true_answer) == final_answer

    return entry

# =============================
# RUN EXPERIMENT
# =============================

def run_experiment(
    csv_path,
    start_index=None,
    limit=None,
    question_column="question",
    answer_column=None
):

    existing = load_existing()

    if start_index is None:
        start_index = get_last_completed_index()

    with open(csv_path, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))

    total = len(rows)
    end_index = total if limit is None else min(start_index + limit, total)

    for i in range(start_index, end_index):

        if i in existing:
            print(f"Skipping Q{i} (already done)")
            continue

        row = rows[i]

        question = row[question_column]
        true_answer = row[answer_column] if answer_column else None

        print(f"\nRunning Q{i}: {question[:80]}")

        entry = run_question(question, true_answer)
        entry["question_id"] = i

        save_entry(entry)

    print("\nExperiment finished.")

In [ ]:
process = subprocess.Popen(["ollama", "serve"])

In [ ]:
run_experiment(
    csv_path="replacement_trivia_questions.csv",
    start_index=0,
    limit=1000,
    answer_column="answer"
)


Running Q0: The body's 3 major types of blood vessels are the arteries, capillaries & these


/tmp/ipykernel_2904/3113255850.py:237: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat(),



Running Q1: Pashtun leader & nonviolence advocated Abdul Ghaffar Khan was known as the "fron

Running Q2: (I'm Louis C.K.) In a 2015 film I played Arlen Hird, a composite character who i

Running Q3: Legend says this capital is named for Kii, one of the 3 brothers who founded it

Running Q4: A broken-down old nag is called a Rosinante in honor of this Spanish character's

Experiment finished.
